<a href="https://colab.research.google.com/github/andyotani/Machine-learning_26-27/blob/main/JS09/Tugas__Lab_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Langkah 1: Instalasi Library yang Dibutuhkan
Kita perlu menginstal library untuk ketiga algoritma ANN (Approximate Nearest Neighbors) tersebut.

In [1]:
!pip install annoy faiss-cpu hnswlib pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 8.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.3 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520034 sha256=cc340e0ba2c2abc2bab61bb5debb41f2bd6975643dc623984289aadbba4012f4
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679801 sha256=c12697dd988ff487614e723d47d1d0d674213f49b69413bae0166beaa1b69224
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

### Langkah 2 & 3: Import Library dan Membuat Fungsi Benchmark
Kita akan membuat skrip pengujian untuk mengukur waktu pembuatan indeks (Build Time) dan waktu pencarian (Query/Search Time) untuk masing-masing algoritma.

In [2]:
import time
import numpy as np
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

def benchmark_ann(n_data, dim, n_queries=100):
    print(f"--- Menjalankan Benchmark untuk {n_data} data dengan {dim} Dimensi ---")

    # Generate data acak (Float32 diperlukan oleh FAISS & HNSW)
    np.random.seed(42)
    data = np.random.random((n_data, dim)).astype('float32')
    queries = np.random.random((n_queries, dim)).astype('float32')
    k = 10 # Mencari 10 tetangga terdekat

    results = {}

    # ================= ANNOY =================
    try:
        start_time = time.time()
        # Menggunakan 'euclidean' metric
        annoy_index = AnnoyIndex(dim, 'euclidean')
        for i in range(n_data):
            annoy_index.add_item(i, data[i])
        annoy_index.build(10) # 10 trees
        annoy_build_time = time.time() - start_time

        start_time = time.time()
        for q in queries:
            annoy_index.get_nns_by_vector(q, k)
        annoy_query_time = time.time() - start_time

        results['ANNOY'] = {
            'Build Time (s)': annoy_build_time,
            'Query Time (s)': annoy_query_time
        }
    except Exception as e:
        results['ANNOY'] = {'Build Time (s)': 'Error', 'Query Time (s)': 'Error'}

    # ================= FAISS =================
    try:
        start_time = time.time()
        # Menggunakan Flat L2 index (Euclidean)
        faiss_index = faiss.IndexFlatL2(dim)
        faiss_index.add(data)
        faiss_build_time = time.time() - start_time

        start_time = time.time()
        faiss_index.search(queries, k)
        faiss_query_time = time.time() - start_time

        results['FAISS'] = {
            'Build Time (s)': faiss_build_time,
            'Query Time (s)': faiss_query_time
        }
    except Exception as e:
        results['FAISS'] = {'Build Time (s)': 'Error', 'Query Time (s)': 'Error'}

    # ================= HNSW =================
    try:
        start_time = time.time()
        # Menggunakan L2 space
        hnsw_index = hnswlib.Index(space='l2', dim=dim)
        hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
        hnsw_index.add_items(data)
        hnsw_build_time = time.time() - start_time

        start_time = time.time()
        hnsw_index.set_ef(50) # Query-time accuracy parameter
        hnsw_index.knn_query(queries, k=k)
        hnsw_query_time = time.time() - start_time

        results['HNSW'] = {
            'Build Time (s)': hnsw_build_time,
            'Query Time (s)': hnsw_query_time
        }
    except Exception as e:
        results['HNSW'] = {'Build Time (s)': 'Error', 'Query Time (s)': 'Error'}

    return results

### Langkah 4: Menjalankan Pengujian
Kita jalankan fungsi benchmark untuk konfigurasi ukuran kecil (1.000 data) dan ukuran besar (1.000.000 data) sesuai dengan kebutuhan tugas Anda.

In [3]:
# Jalankan skenario 1: 1000 data / 2D
res_1k_2d = benchmark_ann(1000, 2)

# Jalankan skenario 2: 1.000.000 data / 5D
# Catatan: Membangun index 1 juta data di HNSW/Annoy membutuhkan waktu beberapa saat
res_1m_5d = benchmark_ann(1000000, 5)

--- Menjalankan Benchmark untuk 1000 data dengan 2 Dimensi ---
--- Menjalankan Benchmark untuk 1000000 data dengan 5 Dimensi ---


### Langkah 5: Penyajian Tabel Hasil
Mari gabungkan hasil pengujian di atas ke dalam format tabel DataFrame yang rapi.

In [4]:
data_summary = []

# Skenario 1
data_summary.append({
    'Jumlah Data/Dimensi': '1000 / 2D',
    'ANNOY (Build/Query s)': f"{res_1k_2d['ANNOY']['Build Time (s)']:.4f}s / {res_1k_2d['ANNOY']['Query Time (s)']:.4f}s",
    'FAISS (Build/Query s)': f"{res_1k_2d['FAISS']['Build Time (s)']:.4f}s / {res_1k_2d['FAISS']['Query Time (s)']:.4f}s",
    'HNSW (Build/Query s)': f"{res_1k_2d['HNSW']['Build Time (s)']:.4f}s / {res_1k_2d['HNSW']['Query Time (s)']:.4f}s"
})

# Skenario 2
data_summary.append({
    'Jumlah Data/Dimensi': '1000000 / 5D',
    'ANNOY (Build/Query s)': f"{res_1m_5d['ANNOY']['Build Time (s)']:.2f}s / {res_1m_5d['ANNOY']['Query Time (s)']:.4f}s",
    'FAISS (Build/Query s)': f"{res_1m_5d['FAISS']['Build Time (s)']:.2f}s / {res_1m_5d['FAISS']['Query Time (s)']:.4f}s",
    'HNSW (Build/Query s)': f"{res_1m_5d['HNSW']['Build Time (s)']:.2f}s / {res_1m_5d['HNSW']['Query Time (s)']:.4f}s"
})

df_hasil = pd.DataFrame(data_summary)
display(df_hasil)

,Jumlah Data/Dimensi,ANNOY (Build/Query s),FAISS (Build/Query s),HNSW (Build/Query s)
0,1000 / 2D,0.0231s / 0.0024s,0.0001s / 0.0008s,0.0508s / 0.0015s
1,1000000 / 5D,25.90s / 0.0055s,0.01s / 0.4939s,161.42s / 0.0064s
